# Projeto 3 — Variational Autoencoder (VAE)

In [1]:
import os
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from ucimlrepo import fetch_ucirepo

DATA_DIR = Path("../data/raw").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

## Dataset 1: Dry Bean
[Fonte: UCI Machine Learning Repository, ID 602](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset). São 13.611 amostras, 16 atributos numéricos e 7 classes de feijão.

In [2]:
dry_bean_path = DATA_DIR / "dry_bean.csv"
if dry_bean_path.exists():
    dry_bean_df = pd.read_csv(dry_bean_path)
else:
    dry_bean = fetch_ucirepo(id=602)
    dry_bean_df = pd.concat(
        [dry_bean.data.features, dry_bean.data.targets], axis=1
    )
    dry_bean_df.to_csv(dry_bean_path, index=False)

X_dry_bean = dry_bean_df.drop(columns="Class").copy()
y_dry_bean = dry_bean_df["Class"].copy()

print(f"Dry Bean: {X_dry_bean.shape[0]} amostras, {X_dry_bean.shape[1]} atributos")
display(X_dry_bean.head())
display(y_dry_bean.value_counts().sort_index().rename("amostras").to_frame())


Dry Bean: 13611 amostras, 16 atributos


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166


,amostras
Class,
BARBUNYA,1322
BOMBAY,522
CALI,1630
DERMASON,3546
HOROZ,1928
SEKER,2027
SIRA,2636


## Dataset 2: MNIST
[Fonte: MNIST pelo Keras](https://keras.io/api/datasets/mnist/). São imagens de dígitos de 0 a 9, em tons de cinza, com 28 × 28 pixels. Preservamos a divisão oficial: 60.000 imagens para treino e 10.000 para teste.

In [3]:
(X_mnist_train, y_mnist_train), (X_mnist_test, y_mnist_test) = (
    keras.datasets.mnist.load_data()
)

print(f"MNIST treino: {X_mnist_train.shape}, rótulos: {y_mnist_train.shape}")
print(f"MNIST teste: {X_mnist_test.shape}, rótulos: {y_mnist_test.shape}")
display(pd.DataFrame({
    "treino": pd.Series(y_mnist_train).value_counts().sort_index(),
    "teste": pd.Series(y_mnist_test).value_counts().sort_index(),
}).rename_axis("dígito"))


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
MNIST treino: (60000, 28, 28), rótulos: (60000,)
MNIST teste: (10000, 28, 28), rótulos: (10000,)


,treino,teste
dígito,,
0,5923,980
1,6742,1135
2,5958,1032
3,6131,1010
4,5842,982
5,5421,892
6,5918,958
7,6265,1028
8,5851,974


## 4. Separação e pré-processamento

Os conjuntos de **treino** ajustam os pesos e os transformadores; os de **validação** orientam a escolha da arquitetura. Os conjuntos de **teste** ficam reservados para a avaliação final. Todas as divisões usam estratificação para manter aproximadamente as proporções das classes e `random_state=SEED`.

- **Dry Bean:** aproximadamente 70% treino, 15% validação e 15% teste. O `StandardScaler` aprende média e desvio somente do treino. Os atributos padronizados podem ser negativos: usaremos saída linear e erro de reconstrução quadrático no VAE deste dataset.
- **MNIST:** mantemos as 10.000 imagens de teste oficiais; das 60.000 imagens de treino, separamos 10.000 para validação. Dividimos os pixels por 255 e convertemos para `float32`, no intervalo `[0, 1]`. Para começar com VAEs densos, cada imagem vira um vetor de 784 pixels; para visualizar, basta usar `.reshape(-1, 28, 28)`.

Os dados brutos das seções anteriores permanecem disponíveis. Os dicionários `dados_dry_bean` e `dados_mnist` terão entradas `X_train`, `X_val`, `X_test`, `y_train`, `y_val` e `y_test`. Os rótulos permanecem originais e serão usados nas análises, sem entrar como alvo de classificação no VAE não supervisionado.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def preparar_dry_bean(X, y, seed=SEED):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=seed
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=seed
    )

    scaler = StandardScaler()
    # fit somente no treino evita vazamento de informação da validação/teste.
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    return {
        "X_train": X_train_scaled,
        "X_val": scaler.transform(X_val).astype(np.float32),
        "X_test": scaler.transform(X_test).astype(np.float32),
        "y_train": np.asarray(y_train),
        "y_val": np.asarray(y_val),
        "y_test": np.asarray(y_test),
        "scaler": scaler,  # inverse_transform recupera a escala original.
        "feature_names": list(X.columns),
    }


def preparar_mnist(X_train_original, y_train_original, X_test_original,
                   y_test_original, seed=SEED):
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_original, y_train_original,
        test_size=1 / 6, stratify=y_train_original, random_state=seed
    )

    def normalizar_e_vetorizar(images):
        return images.reshape(len(images), -1).astype(np.float32) / 255.0

    return {
        "X_train": normalizar_e_vetorizar(X_train),
        "X_val": normalizar_e_vetorizar(X_val),
        "X_test": normalizar_e_vetorizar(X_test_original),
        "y_train": np.asarray(y_train),
        "y_val": np.asarray(y_val),
        "y_test": np.asarray(y_test_original),
        "image_shape": tuple(X_train_original.shape[1:]),
    }

In [5]:
dados_dry_bean = preparar_dry_bean(X_dry_bean, y_dry_bean)
dados_mnist = preparar_mnist(
    X_mnist_train, y_mnist_train, X_mnist_test, y_mnist_test
)

resumo = []
for nome, dados in (("Dry Bean", dados_dry_bean), ("MNIST", dados_mnist)):
    for split in ("train", "val", "test"):
        X, y = dados[f"X_{split}"], dados[f"y_{split}"]
        assert len(X) == len(y)
        assert X.dtype == np.float32
        assert np.isfinite(X).all()
        resumo.append({
            "dataset": nome, "conjunto": split,
            "amostras": len(X), "atributos": X.shape[1],
            "classes": len(np.unique(y)),
        })

display(pd.DataFrame(resumo))

# Acompanhar a estratificação sem usar os rótulos para treinar o VAE.
for nome, dados in (("Dry Bean", dados_dry_bean), ("MNIST", dados_mnist)):
    print(f"Distribuição das classes — {nome}")
    display(pd.DataFrame({
        split: pd.Series(dados[f"y_{split}"]).value_counts(normalize=True)
        for split in ("train", "val", "test")
    }).sort_index().round(4))

,dataset,conjunto,amostras,atributos,classes
0,Dry Bean,train,9527,16,7
1,Dry Bean,val,2042,16,7
2,Dry Bean,test,2042,16,7
3,MNIST,train,50000,784,10
4,MNIST,val,10000,784,10
5,MNIST,test,10000,784,10


Distribuição das classes — Dry Bean


,train,val,test
BARBUNYA,0.0971,0.0970,0.0975
BOMBAY,0.0383,0.0387,0.0382
CALI,0.1198,0.1195,0.1200
DERMASON,0.2605,0.2605,0.2605
HOROZ,0.1417,0.1415,0.1415
SEKER,0.1489,0.1489,0.1489
SIRA,0.1937,0.1939,0.1934


Distribuição das classes — MNIST


,train,val,test
0,0.0987,0.0987,0.0980
1,0.1124,0.1124,0.1135
2,0.0993,0.0993,0.1032
3,0.1022,0.1022,0.1010
4,0.0974,0.0974,0.0982
5,0.0904,0.0903,0.0892
6,0.0986,0.0986,0.0958
7,0.1044,0.1044,0.1028
8,0.0975,0.0975,0.0974
9,0.0991,0.0992,0.1009
